In [ ]:
## Loading PDF

In [1]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader
)

c:\Users\divyankk.I-FLEX\Documents\Langchain\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print("Testing PyPDFLoader...")
try:
    pypdfloader = PyPDFLoader("./data/pdf/Attention.pdf")
    loader = pypdfloader.load()
    print(loader[0].page_content[:500])
except Exception as e:
    print(f"Error loading PDF with PyPDFLoader: {e}")

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


In [6]:
print("\nTesting PyMuPDFLoader...")
try:
    pymupdfloader = PyMuPDFLoader("./data/pdf/Attention.pdf")
    loaderpymupdf = pymupdfloader.load()
    print(len(loaderpymupdf))
    print(loaderpymupdf[0].page_content[:500])
except Exception as e:
    print(f"Error loading PDF with PyMuPDFLoader: {e}")


Testing PyMuPDFLoader...
15
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


In [8]:
## advanced pdf parsing
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [ ]:
## Advanced PDF parsing with PyPDFLoader and RecursiveCharacterTextSplitter
class SmartPdfProcessor:
    def __init__(self, chunk_size = 1000, chunk_overlap = 200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.textSplitter = RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "]
        )
    
    def process_pdf(self, pdf_path):

        ## pdf loader
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()

        ## process each page
        parsed_docs = []

        for page_num,page in enumerate(pages):
            ## clean text
            cleaned_text = self._clean_text(page.page_content)

            ## skip almost empty pages
            if(len(cleaned_text) < 50):
                continue

            ## create chunks with extra metadata
            chunks = self.textSplitter.create_documents(
                texts =[cleaned_text],
                metadatas =[
                    {
                        **page.metadata,
                        "page": page_num + 1,
                        "total_pages": len(pages),
                        "chunk_method": "smart_pdf_loader",
                        "char_count": len(cleaned_text)
                    }
                ]
            )
            parsed_docs.extend(chunks)
        return parsed_docs
    
    def _clean_text(self,text):
        #remove extra whitespace and newlines
        text = " ".join(text.split())

        # fix common PDF extraction issues
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")
        return text



In [18]:
processor = SmartPdfProcessor()

In [19]:
processor

In [23]:
try:
    process_doc = processor.process_pdf("./data/pdf/Attention.pdf")
    print(f"Total Chunks Created: {len(process_doc)}")
    if process_doc:
        for key, value in process_doc[0].metadata.items():
            print(f"{key}: {value}")
except Exception as e:
    print(f"Error processing PDF with SmartPdfProcessor: {e}")

Total Chunks Created: 52
producer: pdfTeX-1.40.25
creator: LaTeX with hyperref
creationdate: 2023-08-03T00:07:29+00:00
author: 
keywords: 
moddate: 2023-08-03T00:07:29+00:00
ptex.fullbanner: This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5
subject: 
title: 
trapped: /False
source: ./data/pdf/Attention.pdf
total_pages: 15
page: 1
page_label: 1
chunk_method: smart_pdf_loader
char_count: 2858
